In [490]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [491]:
df_articles = pd.read_csv('data/articles.csv')
df_customer = pd.read_csv('data/customers.csv')
df_transactions = pd.read_csv('data/transactions.csv')

df_transactions['t_dat'] = pd.to_datetime(df_transactions['t_dat'])

# Load data trong 5 tuần gần nhất để đảm bảo tính chắc chắn của các giao dịch
from datetime import timedelta
max_date = df_transactions['t_dat'].max()
start_date = max_date - timedelta(days=35)  # 5 weeks = 35 days
df_transactions_recent = df_transactions[df_transactions['t_dat'] >= start_date]
df_transactions_recent['week'] = (max_date - df_transactions_recent['t_dat']).dt.days // 7 # Tạo cột 'week' để phân loại giao dịch theo tuần gần nhất ( 0 là tuần gần nhất )

In [492]:
df_transactions_recent

,t_dat,customer_id,article_id,price,sales_channel_id,week
0,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,800691007,0.011847,1,4
1,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,841182001,0.016932,1,4
2,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,624443037,0.016932,1,4
3,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,861600003,0.050831,1,4
4,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,815579001,0.030492,1,4
...,...,...,...,...,...,...
1224819,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,929511001,0.059305,2,0
1224820,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,891322004,0.042356,2,0
1224821,2020-09-22,fff380805474b287b05cb2a7507b9a013482f7dd0bce0e...,918325001,0.043203,1,0
1224822,2020-09-22,fff4d3a8b1f3b60af93e78c30a7cb4cf75edaf2590d3e5...,833459002,0.006763,1,0


In [493]:
numeric_cols = df_articles.select_dtypes(include=['float64', 'int64']).columns.tolist()
categorical_cols = df_articles.select_dtypes(include=['object']).columns.tolist()
print("Numeric columns:", {len(numeric_cols): numeric_cols})
print("Categorical columns:", {len(categorical_cols): categorical_cols})

Numeric columns: {11: ['article_id', 'product_code', 'product_type_no', 'graphical_appearance_no', 'colour_group_code', 'perceived_colour_value_id', 'perceived_colour_master_id', 'department_no', 'index_group_no', 'section_no', 'garment_group_no']}
Categorical columns: {14: ['prod_name', 'product_type_name', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'perceived_colour_value_name', 'perceived_colour_master_name', 'department_name', 'index_code', 'index_name', 'index_group_name', 'section_name', 'garment_group_name', 'detail_desc']}


In [494]:
# Check the number of null values per column (initial status)
missing_counts = df_articles.isnull().sum().sort_values(ascending=False)
print("\nTop columns with most missing values (count):")
print(missing_counts.head(10))


Top columns with most missing values (count):
detail_desc                     416
perceived_colour_master_name      0
garment_group_name                0
garment_group_no                  0
section_name                      0
section_no                        0
index_group_name                  0
index_group_no                    0
index_name                        0
index_code                        0
dtype: int64


In [495]:
# Static Features 
# Giúp model phân loại hàng hóa dựa trên các đặc điểm như nhóm sản phẩm, màu sắc, v.v.
col_to_use = ['article_id', 'product_group_name', 'graphical_appearance_name', 'colour_group_name', 'section_name', 'index_group_name']
df_items = df_articles[col_to_use].copy()

# Label encoding
for col in col_to_use[1:]:  # Bỏ qua 'article_id' vì nó là định danh duy nhất
    df_items[col] = df_items[col].astype('category').cat.codes

In [496]:
def missing_data(df_customer):
    """
    missing_data Computes Percentage of Missing Values for the dataframe

    Parameters
    ----------
    df_customer : dataframe
        Dataframe for which the missing value percentages need to be computed.

    Returns
    -------
    missing_df: dataframe
        Dataframe with missing value percentages.
    """
    total = df_customer.isnull().sum().sort_values(ascending=False)
    percent = (df_customer.isnull().sum() / df_customer.isnull().count() * 100).sort_values(
        ascending=False
    )
    missing_df = pd.concat([total, percent], axis=1, keys=["Total", "Percent"])
    return missing_df

missing_data(df_customer)

,Total,Percent
Active,907576,66.150819
FN,895050,65.237831
fashion_news_frequency,16011,1.167000
age,15861,1.156066
club_member_status,6062,0.441843
customer_id,0,0.000000
postal_code,0,0.000000


In [497]:
# Handle nulls in customers
df_customer['FN'] = df_customer['FN'].fillna(0)
df_customer['Active'] = df_customer['Active'].fillna(0)
df_customer['FN'] = df_customer['FN'].astype(int)
df_customer['Active'] = df_customer['Active'].astype(int) # convert to int for better handling 
df_customer['fashion_news_frequency'] = df_customer['fashion_news_frequency'].fillna('NONE')
df_customer['club_member_status'] = df_customer['club_member_status'].fillna('Non-Member')
df_customer['age'] = df_customer.groupby('club_member_status')['age'].transform(lambda x: x.fillna(x.median())) # Fill age with median age of the respective club_member_status group


df_customer['age_group'] = pd.cut(df_customer['age'], bins=[0, 20, 30, 40, 50, 60, 100], labels=['&lt;20', '20-30', '30-40', '40-50', '50-60', '60+'])      

In [498]:
# User features - keep numeric binary flags and one-hot encode only categorical variables
user_features = pd.get_dummies(
    df_customer[['club_member_status', 'fashion_news_frequency', 'age_group']])

# Retain FN and Active as numeric features
df_customer = pd.concat(
    [df_customer[['customer_id', 'FN', 'Active']], user_features],
    axis=1
)

In [499]:
missing_data(df_transactions)

,Total,Percent
t_dat,0,0.0
customer_id,0,0.0
article_id,0,0.0
price,0,0.0
sales_channel_id,0,0.0


In [500]:
df_transactions_recent

,t_dat,customer_id,article_id,price,sales_channel_id,week
0,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,800691007,0.011847,1,4
1,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,841182001,0.016932,1,4
2,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,624443037,0.016932,1,4
3,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,861600003,0.050831,1,4
4,2020-08-21,001c2513c498432fdff9eb09349ddf02d31826b148a8c9...,815579001,0.030492,1,4
...,...,...,...,...,...,...
1224819,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,929511001,0.059305,2,0
1224820,2020-09-22,fff2282977442e327b45d8c89afde25617d00124d0f999...,891322004,0.042356,2,0
1224821,2020-09-22,fff380805474b287b05cb2a7507b9a013482f7dd0bce0e...,918325001,0.043203,1,0
1224822,2020-09-22,fff4d3a8b1f3b60af93e78c30a7cb4cf75edaf2590d3e5...,833459002,0.006763,1,0


In [501]:
# Recent Popularity ( Độ phổ biến gần đây)
# Vai trò: Model học được "sản phẩm đang trend". Món nào được mua nhiều trong tuần gần nhất có thể sẽ tiếp tục được mua.
# Remove old columns if they already exist (avoid merge conflicts)
df_items = df_items.drop(columns=[c for c in ['pop_1week', 'pop_2week'] if c in df_items.columns], errors='ignore')

# Week 1 popularity
pop_1week = (
    df_transactions_recent[df_transactions_recent['week'] == 1]
    .groupby('article_id')
    .size()
    .reset_index(name='pop_1week')
)

# Week 2 popularity
pop_2week = (
    df_transactions_recent[df_transactions_recent['week'] == 2]
    .groupby('article_id')
    .size()
    .reset_index(name='pop_2week')
)

# Merge
df_items = df_items.merge(pop_1week, on='article_id', how='left')
df_items = df_items.merge(pop_2week, on='article_id', how='left')

# Fill NA only for new columns
df_items[['pop_1week', 'pop_2week']] = df_items[['pop_1week', 'pop_2week']].fillna(0)

# Popularity trend
df_items['popularity_trend'] = (
    (df_items['pop_1week'] - df_items['pop_2week']) / df_items['pop_2week'] + 1)

# Feature 3: Average Price (Giá trung bình của món hàng)
# Vai trò: Xác định phân khúc sản phẩm (Cao cấp hay Bình dân).
avg_price = df_transactions_recent.groupby('article_id')['price'].mean().reset_index(name='item_avg_price')
df_items = df_items.merge(avg_price, on='article_id', how='left')
print(f"Article Features Created: {df_items.shape}")

Article Features Created: (105542, 10)


In [502]:
df_items

,article_id,product_group_name,graphical_appearance_name,colour_group_name,section_name,index_group_name,pop_1week,pop_2week,popularity_trend,item_avg_price
0,108775015,7,25,1,43,2,0.0,0.0,NaN,NaN
1,108775044,7,25,47,43,2,3.0,4.0,0.75,0.008384
2,108775051,7,26,29,43,2,0.0,0.0,NaN,NaN
3,110065001,16,25,1,46,2,0.0,0.0,NaN,NaN
4,110065002,16,25,47,46,2,0.0,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
105537,953450001,13,22,1,38,3,10.0,2.0,5.00,0.016836
105538,953763001,7,25,1,18,2,17.0,0.0,inf,0.021908
105539,956217002,5,25,1,53,2,13.0,0.0,inf,0.059152
105540,957375001,0,25,1,9,1,0.0,0.0,NaN,NaN


In [503]:
# Item popularity 

item_popularity = df_transactions_recent.groupby('article_id').size().reset_index(name='purchase_count')

# Average price per user
user_avg_price = df_transactions_recent.groupby('customer_id')['price'].mean().reset_index(name='avg_price')

# Purchase frequency per user
user_purchase_count = df_transactions_recent.groupby('customer_id').size().reset_index(name='purchase_count')

# Last purchase date per user
user_last_purchase = df_transactions_recent.groupby('customer_id')['t_dat'].max().reset_index(name='last_purchase')

# Recency (days since last purchase in the 5-week window)
max_date = df_transactions_recent['t_dat'].max()
user_last_purchase['recency'] = (max_date - user_last_purchase['last_purchase']).dt.days

# Merge user features
df_customer = df_customer.merge(user_purchase_count, on='customer_id', how='left').fillna(0)
df_customer = df_customer.merge(user_avg_price, on='customer_id', how='left').fillna(0)
df_customer = df_customer.merge(user_last_purchase[['customer_id', 'recency']], on='customer_id', how='left')

# For items, convert article_id to int for merge compatibility
item_popularity['article_id'] = item_popularity['article_id'].astype(int)
df_articles = df_articles.merge(item_popularity, on='article_id', how='left').fillna(0)

print('User features merged. Customer shape:', df_customer.shape)
print('Item features merged. Article shape:', df_articles.shape)
df_customer.head()

User features merged. Customer shape: (1371980, 19)
Item features merged. Article shape: (105542, 26)


,customer_id,FN,Active,club_member_status_ACTIVE,club_member_status_LEFT CLUB,club_member_status_Non-Member,club_member_status_PRE-CREATE,fashion_news_frequency_Monthly,fashion_news_frequency_NONE,fashion_news_frequency_Regularly,age_group_&lt;20,age_group_20-30,age_group_30-40,age_group_40-50,age_group_50-60,age_group_60+,purchase_count,avg_price,recency
0,00000dbacae5abe5e23885899a1fa44253a17956c6d1c3...,0,0,True,False,False,False,False,True,False,False,False,False,True,False,False,1.0,0.050831,17.0
1,0000423b00ade91418cceaf3b26c6af3dd342b51fd051e...,0,0,True,False,False,False,False,True,False,False,True,False,False,False,False,0.0,0.000000,NaN
2,000058a12d5b43e67d225668fa1f8d618c13dc232df0ca...,0,0,True,False,False,False,False,True,False,False,True,False,False,False,False,1.0,0.061000,7.0
3,00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2...,0,0,True,False,False,False,False,True,False,False,False,False,False,True,False,0.0,0.000000,NaN
4,00006413d8573cd20ed7128e53b7b13819fe5cfc2d801f...,1,1,True,False,False,False,False,False,True,False,False,False,False,True,False,0.0,0.000000,NaN


In [504]:
import re

def build_description(row):
    text_parts = []

    # 🔥 BOOST product type (VERY IMPORTANT)
    product_type = str(row['product_type_name']).lower()
    text_parts += [product_type] * 3

    # Other attributes
    text_parts += [
        str(row['prod_name']),
        str(row['product_group_name']),
        str(row['graphical_appearance_name']),
        str(row['colour_group_name']),
        str(row['perceived_colour_master_name']),
        str(row['index_name']),
        str(row['index_group_name']),
        str(row['section_name'])
    ]

    text = ' '.join(text_parts).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)

    return text

df_articles['clothes_description'] = df_articles.apply(build_description, axis=1)

# Lemmatization to reduce words to their base form (e.g., "dresses" -> "dress") to improve matching

from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    words = [lemmatizer.lemmatize(w) for w in words]
    return ' '.join(words)

df_articles['clothes_description'] = df_articles['clothes_description'].apply(lemmatize_text)


synonyms = {
    'baby': ['kids', 'children'],
    'kids': ['baby', 'children'],
    'children': ['baby', 'kids'],
    
    'women': ['ladies', 'female'],
    'ladies': ['women', 'female'],
    
    'men': ['male'],
    'male': ['men']
}

#Add synonyms to the description to increase recall (e.g., "baby" -> "baby kids children")
def add_synonyms(text):
    words = text.split()
    expanded = list(words)  # keep duplicates for weighting
    
    for w in words:
        if w in synonyms:
            expanded.extend(synonyms[w])
    
    return ' '.join(expanded)

df_articles['clothes_description'] = df_articles['clothes_description'].apply(add_synonyms)

# Remove stop words to reduce noise (e.g., "the", "and", "is")
from nltk.corpus import stopwords
stopwords = set(stopwords.words('english'))

df_articles['clothes_description'] = df_articles['clothes_description'].apply(
    lambda x: ' '.join([w for w in x.split() if w not in stopwords])
)

In [505]:
# TF-IDF Vectorization to convert text to numerical features for similarity matching
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=200,
    min_df=2,
    ngram_range=(1,2) 
)

tfidf_matrix = tfidf.fit_transform(df_articles['clothes_description'])

In [506]:
# SVD for dimensionality reduction (optional, can help with noise and speed)
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=40, random_state=42)
item_vectors = svd.fit_transform(tfidf_matrix)

In [509]:
from sklearn.metrics.pairwise import cosine_similarity

def search_items(query):
    # same preprocessing as training
    query = query.lower()
    
    # vectorize query
    query_vec = tfidf.transform([query])
    query_vec = svd.transform(query_vec)
    
    # cosine similarity happens HERE
    similarities = cosine_similarity(query_vec, item_vectors).flatten()
    
    return similarities

In [512]:
similarities = search_items("baby dress")

In [515]:
df_result = df_articles[['article_id', 'product_type_name']].copy()
df_result['similarity'] = similarities

df_result = df_result.sort_values('similarity', ascending=False)
df_result.head(10)

,article_id,product_type_name,similarity
23610,606125001,Dress,0.939543
2940,442992005,Dress,0.939543
91811,851309001,Dress,0.939430
35808,648424001,Dress,0.939430
99082,882261001,Dress,0.939430
101714,898676002,Dress,0.939217
89760,839869003,Dress,0.939217
9011,528247001,Dress,0.938514
37672,654467001,Dress,0.938306
6889,509014006,Dress,0.937351


In [514]:
df_articles_after = pd.concat(
    [
        df_articles[['article_id']].reset_index(drop=True),
        pd.DataFrame(item_vectors, columns=[f'item_feat_{i}' for i in range(40)])
    ],
    axis=1
)

df_articles_after

,article_id,item_feat_0,item_feat_1,item_feat_2,item_feat_3,item_feat_4,item_feat_5,item_feat_6,item_feat_7,item_feat_8,...,item_feat_30,item_feat_31,item_feat_32,item_feat_33,item_feat_34,item_feat_35,item_feat_36,item_feat_37,item_feat_38,item_feat_39
0,108775015,0.256977,0.123224,0.182117,0.134800,0.115048,0.330584,-0.272517,0.451451,-0.076560,...,-0.003757,0.020675,-0.030939,-0.032929,0.020343,0.001697,-0.016634,-0.031154,0.015157,-0.019439
1,108775044,0.243501,0.094106,0.199590,0.086778,0.102783,0.304456,-0.300719,0.466316,-0.114969,...,0.016838,0.030352,-0.017619,-0.043861,0.032046,-0.000673,-0.023561,-0.007140,-0.003068,-0.025533
2,108775051,0.231616,0.087422,0.194854,0.076722,0.097462,0.291445,-0.299801,0.461555,-0.112030,...,0.019361,0.032291,-0.036297,-0.035430,0.030965,-0.019628,-0.024997,-0.017221,-0.003676,-0.044031
3,110065001,0.163508,0.054597,0.097536,0.067041,-0.017602,0.274809,-0.091305,-0.057402,0.453083,...,0.002400,0.027535,-0.044725,0.025168,0.009990,0.046619,0.009960,-0.037406,0.000617,-0.024675
4,110065002,0.150611,0.024456,0.116616,0.013157,-0.029465,0.244874,-0.130197,-0.031272,0.396907,...,0.024973,0.038734,-0.030317,0.008439,0.024389,0.045086,0.003973,-0.015206,-0.015479,-0.028766
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
105537,953450001,0.129945,-0.028981,0.062738,0.053918,-0.019034,0.098656,0.003954,-0.120443,0.264852,...,0.001495,-0.006319,0.032074,0.027463,0.010395,-0.004161,0.031316,-0.038979,0.003499,-0.035363
105538,953763001,0.234672,0.099883,0.172028,0.126585,0.096009,0.292270,-0.226263,0.438255,-0.060290,...,0.008966,0.010925,0.016220,-0.015812,-0.001001,-0.011599,-0.008319,-0.005439,0.014874,0.061004
105539,956217002,0.426227,0.757326,-0.190115,-0.157215,-0.101901,0.013715,-0.149299,-0.167927,0.003798,...,-0.012701,0.029906,-0.010152,-0.019329,-0.029583,0.002073,-0.016091,0.014291,0.014394,0.093532
105540,957375001,0.172466,0.136351,0.080139,0.123986,0.053476,0.141580,0.465099,0.048137,0.005542,...,0.018547,0.054010,-0.008811,-0.031946,0.100815,-0.004668,0.064876,-0.262361,-0.340664,0.284184
